# Rivulet: U. S. EPA Air Quality System
_by Michelle H Wilkerson, Lucas Coletti, and Adelmo Eloy_

### Purpose of this Notebook

This notebook is focused on **Air Quality** as a phenomenon. While most students understand that poor Air Quality can impact health, they may not know that there are many different kinds of air pollution, each caused by different processes and chemicals. These are reflected by different patterns over the course of a day, week, major event, or year.

This data tool allows users to connect to the U. S. Environmental Protection Agency's Air Quality System (AQS) API, which provides air quality data for dates up to six months before the present date (the time lag allows for data to be scientifically validated). You are able to search for air quality data streams in an area of interest, identify a date range of interest within that area, and then access a variety of combinations of data that allow for explorations of different pollutants, comparison of AQI across different regions, and more. The datasets this notebook is constucted to fetch can serve as a launching point for examining what air quality is, and what are its underlying mechanistic and compositional complexities.

<details>
    <summary>Click here for more information about APIs</summary>

API stands for **Application Programming Interface**. Think of it as a language to communicate with data centers so you can search and get the data you need. For students, using APIs is like having a direct line to the most advanced scientific sensors on the planet. It allows us to work with the same **live, real-world data** that professional scientists use to track what's going on in the world around us.

You are welcome to modify and adapt this script. You may find the AQS API documentation [here](https://aqs.epa.gov/aqsweb/documents/data_api.html) and the `pyaqsapi` documentation [here](https://usepa.github.io/pyaqsapi/) helpful.

This notebook was developed as part of NSF Grant 2445609 to support accessing and processing public datasets for middle and high school classroom activities. It's written to be relatively accessible to beginners, but if you have not interacted with computational notebooks or Python before, you may find navigating this tool difficult. (Check out the Show Your Work project for a gentle introduction to computational notebooks for educators!)

Our project is focused on supporting data analysis and mechanistic reasoning in science education. In other words, we want students to learn how data provides information about _how scientific mechanisms work_ and how understanding scientific mechanisms can help them to _explain and interpret patterns in data_. This builds on a long history of research on complex systems and agent-based modeling, and more closely connects that work to current expansions of data analysis across subjects.

# Part I: Setup

First, you need to connect to the API and specify what region and time period you are interested in. This section will help you with that.

### Connecting with AQS

Before you get started, you will need an AQS API Key. To get one, use the url `https://aqs.epa.gov/data/api/signup?email=myemail@example.com` (replace myemail@example.com with your email) and paste it into a browser. You will receive a cute sounding API key via email to the address you provided in the URL. 

Copy your API key and change EMAIL and API_KEY in the cell block below to your email and key from the service. 

You can run the notebook using the test email and key that are already provided, however, the test account has a limited number of uses per day and may not work. Register for an account as soon as you know you'd like to use the service. If you lose your key and need a new one, use the same url with the same email address.

In [ ]:
# EDIT BELOW: Depending on what tool you are using to run this 
# notebook, you may need to replace the "%"" below with "!"
!pip install pyaqsapi # This installs the AQS API package from the EPA

EMAIL = "test@aqs.api" # EDIT HERE: with your registered email
API_KEY = "test" # EDIT HERE: with your API key

The `pyaqsapi` package provides our requested data as pandas dataframes. Below, we run a query to see if the service is available. We've found this service in particular to be finicky at times. If you don't get a successful response on a query, take a break and try again later. Remember that you are more likely to get a successful response if you use your own API key rather than the test key provided here.

<div class="alert alert-block alert-success">
    <b>NOTE:</b> If you are running this notebook in Google CoLab, uncomment the first line of the code below. There will also be lines to uncomment in each "Data Fetch" section, so that you can directly download the .csv files when you are ready to get them.
</div>

In [ ]:
# from google.colab import files ## UNCOMMENT THIS LINE IF USING COLAB
import pyaqsapi as aqs # library for getting data
import pandas as pd # library for working with data

aqs.aqs_credentials(username=EMAIL, key=API_KEY)

aqs.aqs_is_available()

### Customizing Your Location

This section allows you to specify a location and a date for which you would like to collect data. You'll need to know the approximate longitude and latitude of the region you are interested in. One easy way to do this is by asking Google, "What is the longitude and latitude of [area]?" You will use the code in this section to use a minimum and maximum latitude and longitude to draw a bounding box around your location of interest. This will filter your future queries to focus only on air quality sensors found within that box. If you don't find any sensors, select a different location or increase the bounding box size.

Now, identify the specific location you want to explore. AQS can fetch all the air quality data within a bounding box. Let's not get crazy - start with a relatively small bounding box (try one degree for an urban area) and get bigger if you need to. 

(Tip: If you click on a location in Google Maps, you'll see the lat and long for that point in the URL.)

In [ ]:
# Define a bounding box around your target region. 
# If it is densely populated, we suggest you start with 
# a bounding box that is only one degree in area. 

min_lat = 37 # EDIT HERE
max_lat = 38 # EDIT HERE
min_long = -122.5 # EDIT HERE
max_long = -121.5 # EDIT HERE

Below, we'll create a map with the box you defined, to make sure you're capturing what you want.

In [ ]:
%pip install folium #install the mapping library
import folium

bbox = [[min_lat, min_long], [max_lat, max_long]]

# Calculate the center of the box to position the map
map_center = [(bbox[0][0] + bbox[1][0]) / 2, (bbox[0][1] + bbox[1][1]) / 2]

# Create a Folium map object
m = folium.Map(location=map_center, zoom_start=8)

# Add a rectangle for the bounding box to the map
folium.Rectangle(
    bounds=bbox,
    color="#ff0000",        # Red border
    fill=True,
    fill_color="#ff7800",   # Orange fill
    fill_opacity=0.2
).add_to(m)

m

### Customizing Your Date

Next, specify a "target date." Different parts of the notebook will collect different time periods of data (usually about a week's worth to a month's worth of data). This part makes sure that the date you choose is included in the dataset. You may want to consider when a major air quality event (like a wildfire, fireworks, weather event, etc.) happened, or you can choose what you understand to be a "typical" day to get baseline datasets for your region.

In [ ]:
from datetime import datetime, timedelta

target_date = "04-10-2020" # EDIT HERE
target_datetime = datetime.strptime(target_date, "%m-%d-%Y")

### Defining Pollutants

AQS has settings for different collections of parameters that reflect different "classes" of interest. For example, one parameter is SCHOOL AIR TOXICS, which lists 125 industrial air pollutants that have been tracked near some K-12 schools, or HAZARDOUS AIR POLLUTANTS with a total of 407 pollutants. 

We want to start simple here, and focus on the pollutants that are most well-known and most likely to be monitored across the country. This also helps us make sure we're not taxing the system with big queries when we don't need to. AQI POLLUTANTS includes codes for the "big five" pollutants that are used to calculate the Air Quality Index: 

In [ ]:
try:
    parameter_list = aqs.aqs_parameters_by_class("AQI POLLUTANTS")
except Exception as e:
    print(f"Something didn't work: {e}")

parameter_list

You can use this notebook to explore other pollutants by replacing the "codes" you see above with codes for the pollutant you're interested in. 

In each section below, we will suggest different combinations of pollutants to explore. You can always come back here to get the code of other pollutant(s) you might want to explore.

# Part II: Fetch Data

Now, we are ready to begin fetching data for your investigation. Each section below allows you to download a _different kind of dataset_, designed to support exploration of _different patterns and issues related to air quality_. You can scroll through to learn more about each one, or click a link to go directly to a section.

[Exploring Pollutant Behaviors: Ozone and PM2.5](#pollutants)
[Reconstructing AQI](#reconstructing)
[Comparing Neighbors](#comparing)

## Exploring Pollutant Behaviors: Ozone and PM2.5
<a id="pollutants"></a> Data can highlight how different air pollutants have different sources and behaviors. For example, Ozone often peaks in the heat of the afternoon (a photochemical reaction), while PM2.5 (smoke and dust) might stay high all night or spike during morning traffic. 

Here, we guide you through downloading a dataset that you can use to compare the behavior of PM2.5 and Ozone, or other pollutants that you might be interested in.

In [ ]:
PARAMETERS = ('44201', #ozone
              '88101') #pm2.5

### Choose Your Site and Length of Time

First, let's look for the monitoring stations in your custom region that were actually monitoring the pollutants defined above during the time you are interested in.

In [ ]:
bdate = target_datetime
edate = target_datetime + timedelta(days=1)

monitors = aqs.bybox.monitors(
    parameter= ','.join(PARAMETERS), # all six pollutants are listed
    bdate=bdate, 
    edate=edate,
    minlat=min_lat,
    maxlat=max_lat,
    minlon=min_long,
    maxlon=max_long,
)

# Filter the monitors so we are showing only the ones that have all requested parameters
relevant_monitors = monitors.groupby(
    ['state_code','county_code','site_number']).filter( # make groups of each monitor
    lambda group: set(group['parameter_code']) == set(PARAMETERS) # include only groups with both monitors
)


relevant_monitors[['state_code','county_code','site_number','address','city_name','county_name','state_name']].drop_duplicates()

Hopefully, at least one monitor in your designated area appeared that has all the pollutants you are looking for. If not, you may want change the pollutants you are looking for (some pollutants, like SO or PM10, and rarely monitored compared to PM2.5 or Ozone).

We can also adjust how many days of data you will get. This might be helpful depending on the kind of event you want to explore (for example, you may only need a few days to explore the impacts of fireworks on air quality, versus a few weeks if you are tracking the impacts of a longer-lasting wildfire). In the code block below, we specify how many weeks of data we'd like to fetch around our target date. By default, we have set this to fetch two weeks of data, with your target date right in the middle of the time period.

In [ ]:
beginningdate = target_datetime - timedelta(weeks=1)
enddate = target_datetime + timedelta(weeks=1)

Now, use list of sites above to find the monitoring station you want to use to get your data. Enter the corresponding stateFIPS (state_code), countycode, and sitenum (site_number) to get your data.

By default, we have set the code below to get data from a monitoring station in Concord, CA that we know has been monitoring all five AQI pollutants.

### Fetch Your Data

In [ ]:
# CUSTOMIZE THIS: Enter the stateFIPS, county_code, and sitenum of your preferred 
# monitoring station from the list above. 

aqdata = aqs.bysite.sampledata(parameter=','.join(PARAMETERS),
                      bdate=beginningdate,
                      edate=enddate,
                      stateFIPS="06", # EDIT HERE enter your state_code 
                      countycode="001", # EDIT HERE enter your county_code 
                      sitenum="0013") # EDIT HERE enter your site_number 

aqdata

The data we get from the AQS has dates and times, but not a combined datetime field. This would make it hard for us to analyze our data from hour to hour (for example, there would not be an easy way for graphing programs to know that 11pm Dec 31, 2020 comes before 1am on Jan 1, 2021), so let's combine the information so dates and times are all organized correctly.

In [ ]:
aqdata['datetime_local'] = aqdata.apply(lambda row: datetime.strptime(f"{row['date_local']} {row['time_local']}", "%Y-%m-%d %H:%M"), axis=1)

Now, we can graph the data to see if it's demonstrating patterns that are worth exploring further with your students.

### Plot to Explore the Patterns

In [ ]:
import seaborn as sns # a library for graphing

g = sns.FacetGrid(aqdata, 
                  row="parameter_code", 
                  aspect=4, # this makes the graphs 4x wider than they are tall
                  sharey=False) # this treats the scale for each pollutant separately

g.map(sns.lineplot, "datetime_local", "sample_measurement")

### Export for Further Investigation

If this set of data looks interesting enough to keep working with, use the code below to export to csv. We first pivot the data so that it's in a format that's easier to work with using student-friendly tools like Excel or CODAP.

In [ ]:
aqdata_pivoted = aqdata.pivot_table(
    index='datetime_local', # Pivot so each row is a datetime
    columns='parameter_code',
    values='sample_measurement'
).reset_index()

aqdata_pivoted

aqdata_pivoted.to_csv("pollutantlevels.csv") 

# files.download("pollutantlevels.csv") ## UNCOMMENT THIS LINE IF USING COLAB

## Reconstructing AQI from Pollutant Levels
<a id="reconstructing"></a> The Air Quality Index (AQI) is a **composite index** used to communicate how clean or polluted the air is. Instead of tracking just one chemical, it looks at a set of five main pollutants: Ozone, Carbon Monoxide, Sulfur Dioxide,  Nitrogen Dioxide, and particulate matter (PM2.5 and PM10). 

Generating a dataset that includes all of these pollutants is pedagogically powerful because different pollutants have different **"signature patterns."** By reconstructing the AQI ourselves, we can see exactly which pollutant is "driving" the risk level at any given time.

<img src="https://cap-mzansi.com/wp-content/uploads/2023/06/shutterstock_1657303105-1536x602.webp" alt="alt text" width="500">

### Select all AQI Pollutants
In the block below, we define the `PARAMETERS` we want to fetch. By default, we include all six major pollutants that are tracked by the AQI measure. 

In [ ]:
# CUSTOMIZE THIS: Identify the parameters that you want to include in your dataset.

PARAMETERS = (
    '44201', # Ozone
    '88101', # PM2.5
    '42101', # Carbon monoxide
    '42401', # Sulfur dioxide
    '42602', # Nitrogen dioxide (NO2)
    '81102'  # PM10
)

### Find Monitoring Stations that Measure All Pollutants
To generate that most accurate AQI calculation, we need to find a station that reports **all six** the pollutants that are tracked. The code below searches the local region you defined above for these "complete" monitoring stations. 

Unfortunately, not many monitoring stations track every pollutant. It's pretty tricky to find a monitoring station that has all the AQI pollutants in practice. If you can't find one for your area, try reducing the parameters above. Usually, the pollutants that are most important for setting the overall AQI are PM2.5, Ozone, and NO2. Try adjusting the required parameter list in the code above to find a "good enough" monitoring station for your region.

In [ ]:
# RUN THIS: This code figures out what monitoring stations have the information you need.
# If there are no monitoring stations in your area that have all the pollutants, use the
# box above to reduce the number of pollutants you require.

monitors = aqs.bybox.monitors(
    parameter= ','.join(PARAMETERS), # all six pollutants are listed
    bdate=target_datetime, 
    edate=target_datetime + timedelta(days=1),
    minlat=min_lat,
    maxlat=max_lat,
    minlon=min_long,
    maxlon=max_long,
)

# Filter the monitors so we are showing only the ones that have all requested parameters
relevant_monitors = monitors.groupby(
    ['state_code','county_code','site_number']).filter( # make groups of each monitor
    lambda group: set(group['parameter_code']) == set(PARAMETERS) # include only groups with both monitors
)


relevant_monitors[['state_code','county_code','site_number','address','city_name','county_name','state_name']].drop_duplicates()

### Fetch the Time-Series Data
Now we zoom in on a specific station. We are fetching a **two-week window** of data around our target date. This allows students to see several day/night cycles (diurnal patterns), which helps them distinguish between consistent biological/physical rhythms and unique weather events. You can adjust the time period using the next code block.

In [ ]:
# CUSTOMIZE THIS: Define the period of time for which you would like to fetch 
# data. By default, the code below specifies a beginning date one week before
# the target date, and one week after.

beginningdate = target_datetime - timedelta(weeks=1)
enddate = target_datetime + timedelta(weeks=1)

In the following block of code, we fetch data from the monitoring station of your choice. Adjust the `stateFIPS` (state_code), `countycode`, and `sitenum` to match a monitoring station from the list above that has all the pollutants you are looking for.

In [ ]:
# CUSTOMIZE THIS: Enter the stateFIPS, county_code, and sitenum of your preferred 
# monitoring station from the list above. 

aqdata = aqs.bysite.sampledata(parameter=','.join(PARAMETERS),
                      bdate=beginningdate,
                      edate=enddate,
                      stateFIPS="06", # enter the state_code here
                      countycode="013", # enter the county_code here
                      sitenum="0002") # enter the site_number here

aqdata.head()

### Translate Pollutant Levels to the Composite AQI
The next code blocks contain the EPA's official **"breakpoints"** and the functions needed to translate raw concentrations (like ppm or µg/m³) into the 0-500 AQI scale. This illustrates how scientists translate complex measurements into simpler measurements to communicate about physical phenomena. To learn more about this conversion, you can check out the EPA's technical documentation [here](https://document.airnow.gov/technical-assistance-document-for-the-reporting-of-daily-air-quailty.pdf).

In [ ]:
# RUN THIS: Define the EPA Air Quality Index (AQI) Breakpoints
# Format: (AQI_Low, AQI_High, Conc_Low, Conc_High)

AQI_BREAKPOINTS = {
    # PM2.5 (24-hr, in µg/m³)
    '88101': [
        (0, 50, 0.0, 12.0),
        (51, 100, 12.1, 35.4),
        (101, 150, 35.5, 55.4),
        (151, 200, 55.5, 150.4),
        (201, 300, 150.5, 250.4),
        (301, 400, 250.5, 350.4),
        (401, 500, 350.5, 500.4),
    ],
    # PM10 (24-hr, in µg/m³)
    '81102': [
        (0, 50, 0, 54),
        (51, 100, 55, 154),
        (101, 150, 155, 254),
        (151, 200, 255, 354),
        (201, 300, 355, 424),
        (301, 400, 425, 504),
        (401, 500, 505, 604),
    ],
    # O3 (8-hr, in ppm)
    '44201': [
        (0, 50, 0.000, 0.054),
        (51, 100, 0.055, 0.070),
        (101, 150, 0.071, 0.085),
        (151, 200, 0.086, 0.105),
        (201, 300, 0.106, 0.200),
        # Note: AQI > 200 for 8-hr O3 is calculated using 1-hr O3.
        # This implementation assumes you will provide the 8-hr value
        # and will cap at the 201-300 range.
    ],
    # O3 (1-hr, in ppm) - Used when 8-hr values are high TODO
    'O3_1hr': [
        (101, 150, 0.125, 0.164),
        (151, 200, 0.165, 0.204),
        (201, 300, 0.205, 0.404),
        (301, 400, 0.405, 0.504),
        (401, 500, 0.505, 0.604),
    ],
    # CO (8-hr, in ppm)
    '42101': [
        (0, 50, 0.0, 4.4),
        (51, 100, 4.5, 9.4),
        (101, 150, 9.5, 12.4),
        (151, 200, 12.5, 15.4),
        (201, 300, 15.5, 30.4),
        (301, 400, 30.5, 40.4),
        (401, 500, 40.5, 50.4),
    ],
    # SO2 (1-hr, in ppb)
    '42401': [
        (0, 50, 0, 35),
        (51, 100, 36, 75),
        (101, 150, 76, 185),
        (151, 200, 186, 304),
        (201, 300, 305, 604),
        (301, 400, 605, 804),
        (401, 500, 805, 1004),
    ],
    # NO2 (1-hr, in ppb)
    '42602': [
        (0, 50, 0, 53),
        (51, 100, 54, 100),
        (101, 150, 101, 360),
        (151, 200, 361, 649),
        (201, 300, 650, 1249),
        (301, 400, 1250, 1649),
        (401, 500, 1650, 2049),
    ],
}

The code below takes all the thresholds that are defined above, and creates functions that will use them to calculate the AQI for the data we have fetched. These functions will become the machinery we apply to the data to figure out the overall composite AQI.

In [ ]:
import math # we need this library for some of the calculations below

def calculate_individual_aqi(pollutant_code, concentration):        
    if concentration is None or concentration < 0 or \
       not math.isfinite(concentration):
        return None

    C_p = concentration

    # Find the correct breakpoint category
    table = AQI_BREAKPOINTS[pollutant_code]
    
    for (I_lo, I_hi, C_lo, C_hi) in table:
        if C_lo <= C_p <= C_hi:
            # Avoid division by zero if C_hi == C_lo
            if (C_hi - C_lo) == 0:
                aqi = I_lo
            else:
                # Apply the linear interpolation formula
                aqi = ((I_hi - I_lo) / (C_hi - C_lo)) * (C_p - C_lo) + I_lo
                
            return round(aqi)
            
    # If concentration is beyond the highest breakpoint,
    # use the last category for calculation
    (I_lo, I_hi, C_lo, C_hi) = table[-1]
    if C_p > C_hi:
        if (C_hi - C_lo) == 0:
            aqi = I_lo
        else:
            aqi = ((I_hi - I_lo) / (C_hi - C_lo)) * (C_p - C_lo) + I_lo
        return round(aqi)

def calculate_composite_aqi(concentration_data):
    individual_aqis = []
    
    for pollutant_code, concentration in concentration_data.items():
        aqi = calculate_individual_aqi(pollutant_code, concentration)
        
        if aqi is not None:
            individual_aqis.append(aqi)
            
    # The composite AQI is the highest of the available individual AQIs
    if not individual_aqis:
        return None
        
    return max(individual_aqis)

# Helper to build pollutant -> concentration dict and compute composite AQI
def _row_to_composite(row):
    conc = {str(code): row[code] for code in row.index if pd.notnull(row[code])}
    return calculate_composite_aqi(conc)

### Calculate the Composite AQI
Now we apply our "engine" to the raw data. We first calculate the AQI for each individual pollutant at every time point. 

In [ ]:
aqdata['individual_aqi'] = aqdata.apply(
    lambda row: calculate_individual_aqi(row['parameter_code'], row['sample_measurement']),
    axis=1
)

The data we get from the AQS has dates and times, but not a combined datetime field. This would make it hard for us to analyze our data from hour to hour (for example, there would not be an easy way for graphing programs to know that 11pm Dec 31, 2020 comes before 1am on Jan 1, 2021), so let's combine the information so dates and times are all organized correctly.

In [ ]:
combined_series = aqdata['date_local'] + ' ' + aqdata['time_local']
aqdata['datetime_local'] = pd.to_datetime(combined_series, format="%Y-%m-%d %H:%M")

Now, we are going to pivot the dataset so that it is easier to compare pollutants that were measured at the same time. This is what we need to do in order to figure out the composite AQI for each moment of time when pollutants were measured.

In [ ]:
# Pivot so each row is a datetime and columns are pollutant codes with their measurements
measurements = aqdata.pivot_table(
    index='datetime_local',
    columns='parameter_code',
    values='sample_measurement',
)

Now, we find the "highest" value across all pollutants to determine the **Composite AQI**. The code below also does some cleanup to make sure dates are always formatted in the same way and everything is labeled properly.

In [ ]:
# Compute composite AQI for each datetime
measurements['composite_aqi'] = measurements.apply(_row_to_composite, axis=1)

# Make a tidy timeseries dataframe
aqi_timeseries = measurements[['composite_aqi']].reset_index().sort_values('datetime_local')

new_rows = pd.DataFrame({
    'datetime_local': aqi_timeseries['datetime_local'],
    'sample_measurement': aqi_timeseries['composite_aqi'],
    'parameter_code': 'COMPOSITE_AQI',     # tag so these rows are identifiable
    'parameter': 'Composite AQI',
    'individual_aqi': aqi_timeseries['composite_aqi']
})

# Concatenate and keep a consistent datetime dtype; sort by datetime if desired.
aqdata = pd.concat([aqdata, new_rows], ignore_index=True, sort=False)
aqdata['datetime_local'] = pd.to_datetime(aqdata['datetime_local'])
aqdata = aqdata.sort_values('datetime_local').reset_index(drop=True)

### Visualize Patterns
Now, we can plot all our pollutants and the composite AQI on the same time scale. 

This is a great way to see which pollutant is the "limiting factor" or the primary risk driver at different times of the day. For example, you might see the composite score being driven by Ozone in the afternoon and PM2.5 at night.

These visual patterns can raise interesting questions about AQI: Does a score of 50 always mean the same thing? Why or why not? What are the relationships between the different pollutants? 

In [ ]:
# RUN THIS: Plot a grid of reported levels for each pollutant, plus composite AQI

import seaborn as sns

# All that dataframe action messes us the indices. 
# We need to reset the index to visualize everything together.
aqdata = aqdata.reset_index()

# Let's also sort the aqdata so that COMPOSITE_AQI is at the top of the grid.
aqdata = aqdata.sort_values("parameter_code", ascending=False)

g = sns.FacetGrid(aqdata, 
                  row="parameter_code", 
                  aspect=4, # this makes the graphs 4x wider than they are tall
                  sharey=False) # this treats the scale for each pollutant separately

g.map(sns.lineplot, "datetime_local", "sample_measurement")

### Export for Further Investigation
If the patterns look interesting and you'd like to use this dataset with your students, you can export the data to a `.csv` file. This file can then be uploaded to tools like CODAP or Excel for further analysis.

In [ ]:
# Let's pivot so all measures are collected in one row with the datetime 
aqdata = aqdata.reset_index().pivot_table(index='datetime_local', columns='parameter_code', values='sample_measurement')

aqdata.to_csv("aqipluspollutantlevels.csv")

# files.download("aqipluspollutantlevels.csv") ## UNCOMMENT THIS LINE IF USING COLAB

## Comparing Air Quality Between Neighbors
<a id="comparing"></a> Another interesting activity to do with air quality data is to explore differences in air quality between different locations in the same region. This may be useful to think about the impacts of particular natural (e.g. a coastal breeze or mountains) or man-made features (e.g. the presence of a freeway or factory) on local air quality patterns over longer periods of time.

This section helps you conduct a search within the region you defined in Section 1, specifically for finding monitoring stations with the largest differences in mean concentrations between pollutants. 

### Specify Your Pollutant and Time Interval

First, let's identify the specific pollutant you want to compare across sites. Here, we look at PM2.5.

In [ ]:
pollutants = "88101" # EDIT HERE to identify the pollutant you want to compare.

Now, let's define a time period to compare. Below, our default is to look at data for a one-month period around the target date you specified in the setup.

In [ ]:
# define the month around the target datetime
bdate = target_datetime - timedelta(weeks=2)
edate = target_datetime + timedelta(weeks=2)

### Find Neighboring Monitoring Stations

Let's find the monitoring sites within the specified region that have the most dramatic differences in mean pollutant concentrations.

The code below takes daily summaries for each site in the region, for the period of time specified above.

In [ ]:
# RUN THIS: For all sites within the bounding box
# get the aggregated stats by site of the requested pollutant(s) 

aqsummary = aqs.bybox.dailysummary(parameter=pollutants,
                      bdate=bdate,
                      edate=edate,
                      minlat=min_lat,
                      maxlat=max_lat,
                      minlon=min_long,
                      maxlon=max_long)

aqsummary.head()

### Compare and Map Means Between Neighbors

Now, we'll compute overall arithmetic mean of those daily summaries above for each site. We'll then rank the sites by mean to see where we might find some extreme differences. This isn't the only information we need, but it's a good first step for now.

In [ ]:
# sort the sites from highest to lowest mean pollutant concentration.
meanaq = aqsummary.groupby(
    ['state_code', 'county_code', 'site_number']
).mean(numeric_only=True).sort_values(
    by="arithmetic_mean", 
    ascending=False
)

meanaq[['arithmetic_mean']]

Lets map these sites and color each site by its relative mean pollutant concentration. This might help you find out more about which specific regional differences for this pollutant might be most interesting or compelling for students.

In [ ]:
import branca.colormap as cm # for coloring map markers

linear_scale = cm.LinearColormap(
    ["green", "yellow", "red"],
    vmin=min(meanaq['arithmetic_mean']), vmax=max(meanaq['arithmetic_mean'])
)

map_center = [meanaq['latitude'].mean(), meanaq['longitude'].mean()]
m = folium.Map(location=map_center, zoom_start=8)

# 3. Loop through the DataFrame and add markers
for index, row in meanaq.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=10,
        tooltip= index, # Show the name on hover
        fill_color=linear_scale(row['arithmetic_mean']),
        fill_opacity=1
    ).add_to(m)

m

### Choose Focal Neighbors

Okay, now that you've got your map and table, it's time to pick your favorites! Look for two or three sites that are close to each other but might have different surroundings (like one near a freeway and another in a park). 

Copy the `state_code`, `county_code`, and `site_number` for your chosen sites and plop them into the list below. We'll pull the full month of data for these neighbors so we can see how they compare day-to-day. This dataset is great for comparing distributions or time-series patterns.

In [ ]:
# EDIT HERE: Choose 2-3 sites from the table above to compare.
# Format: [state_code, county_code, site_number]
my_sites = [
    ['06', '087', '1005'], 
    ['06', '013', '1004']
]

# make a list of all the columns that will be returned
aqs_columns = [
    "state_code", "county_code", "site_number", "parameter_code", "poc",
    "datum", "parameter_name", "sample_duration", "pollutant_standard",
    "date_local", "time_local", "date_gmt", "time_gmt", "sample_measurement",
    "units_of_measure", "sample_frequency", "detection_limit", "uncertainty",
    "qualifier", "method_type", "method_code", "method_description",
    "cbsa_name", "state_name", "county_name", "site_address", "local_site_name",
    "date_of_last_change", "latitude", "longitude"
]

neighbor_comparison = pd.DataFrame(columns=aqs_columns)

for site in my_sites:
    neighbor_data = aqs.bysite.sampledata(parameter=pollutants,
                        bdate=bdate,
                        edate=edate,
                        stateFIPS=site[0], # enter the state_code here
                        countycode=site[1], # enter the county_code here
                        sitenum=site[2]) # enter the site_number here

    neighbor_comparison = pd.concat([neighbor_comparison, neighbor_data], ignore_index=True)

neighbor_comparison.head()

### Visualize Distributions of Pollutants per Neighbor

Let's take a look at what we have! 

In [ ]:
import matplotlib.pyplot as plt

neighbor_comparison['Site_ID'] = (neighbor_comparison['state_code'].astype(str) + "-" + 
                 neighbor_comparison['county_code'].astype(str) + "-" + 
                 neighbor_comparison['site_number'].astype(str))

plt.figure(figsize=(12, 6))

# Boxplot
sns.boxplot(data=neighbor_comparison, x='Site_ID', y='sample_measurement', color='white')

# Overlay Dots (Stripplot)
sns.stripplot(data=neighbor_comparison, x='Site_ID', y='sample_measurement', color='blue', alpha=0.4, jitter=True)

plt.xticks(rotation=45) # Rotate labels so they don't overlap
plt.title('Distribution by Hierarchical Site ID')
plt.show()

### Download the Dataset

If you think the distributions you see above are worth more investigation -- to see what's happening over time, to see if the differences are statistically significant, or even to do some further work to try and understand why these differences exist, go ahead and download the dataset below!

In [ ]:
# Let's pivot so all measures are collected in one row with the datetime 
neighbor_comparison = neighbor_comparison.pivot(index='datetime_local', columns='parameter_code', values='sample_measurement')

neighbor_comparison.to_csv("neighborcomparison.csv")

# files.download("neighborcomparison.csv") ## UNCOMMENT THIS LINE IF USING COLAB

# Credits

This notebook was developed as part of NSF Grant 2445609 to support accessing and processing public datasets for middle and high school classroom activities.

Data provided by the U.S. Environmental Protection Agency Air Quality System (AQS). You can learn more about AQS data [here](https://aqs.epa.gov/aqsweb/documents/about_aqs_data.html).

Special thanks to the authors of the `pyaqsapi` package for making this data so accessible to the Python community.

Developed by Michelle H Wilkerson, Lucas Coletti, and Adelmo Eloy as part of the Rivulet Project.